# Experiment Runner Parity Checks

This notebook runs `experiment_runner.py` with fully explicit parameters and compares the output against existing notebook-generated CSV files.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from experiment_runner import ExperimentConfig, run_experiment

pd.set_option('display.max_columns', 120)


In [3]:
def compare_to_reference(runner_df, reference_csv):
    reference = pd.read_csv(reference_csv)
    keys = ['method', 'kernel', 'split']
    value_cols = ['mean_relative_error', 'std_relative_error']
    param_cols = [
        'length_scale', 'nu', 'alpha',
        'coef_length_scale', 'coef_nu', 'state_length_scale', 'state_nu',
        'n_train', 'n_eval', 'pca',
    ]
    cols = keys + [col for col in value_cols + param_cols if col in runner_df.columns and col in reference.columns]
    merged = runner_df[cols].merge(reference[cols], on=keys, suffixes=('_runner', '_reference'))
    for col in value_cols:
        if f'{col}_runner' in merged.columns:
            merged[f'{col}_diff'] = merged[f'{col}_runner'] - merged[f'{col}_reference']
            merged[f'{col}_abs_diff'] = merged[f'{col}_diff'].abs()
    return merged


def show_parameter_comparison(merged):
    parameter_columns = [
        'length_scale', 'nu', 'alpha',
        'coef_length_scale', 'coef_nu', 'state_length_scale', 'state_nu',
        'n_train', 'n_eval', 'pca',
    ]
    available = []
    for col in parameter_columns:
        runner_col = f'{col}_runner'
        reference_col = f'{col}_reference'
        if runner_col in merged.columns and reference_col in merged.columns:
            available.extend(['method', 'kernel', 'split', runner_col, reference_col])
    if available:
        display(merged.loc[:, list(dict.fromkeys(available))])


## Framework2 Conservation Law: Sample Mode

This should match `Framework2/results_Conservation_law_sample_no_pca.csv`.


In [4]:
framework2_conservation_config = ExperimentConfig(
    task_mode='sample',
    pde_name='Conservation_law',
    base_dir='Framework2',
    results_dir='.',
    data_subdir='dataset_simple',
    data_filename='solutions.h5',
    ood_filename='ood.h5',
    use_ood=True,
    data_key='data',
    coeff_key='coeffs',
    input_time_index=0,
    channel_index=0,
    train_size=10000,
    train_fraction=0.8,
    alpha=1e-10,
    use_pca=False,
    use_x_pca=True,
    use_coef_pca=True,
    use_y_pca=True,
    y_pca_by_time=True,
    n_x_pca=5,
    n_coef_pca=4,
    n_y_pca=5,
    vanilla_rbf_length_scale=100,
    vanilla_matern_length_scale=1,
    vanilla_matern_nu=5/2,
    product_matern_coef_length_scale=1,
    product_matern_coef_nu=5/2,
    product_matern_state_length_scale=100,
    product_matern_state_nu=3/2,
    product_rbf_coef_length_scale=1,
    product_rbf_state_length_scale=100,
    methods=('vanilla_rbf', 'vanilla_matern', 'method_matern', 'method_rbf'),
    save_csv=True,
    results_filename='parity_Framework2_Conservation_law_sample_no_pca.csv',
)

runner_f2_conservation = run_experiment(framework2_conservation_config)
display(runner_f2_conservation[['method', 'kernel', 'split', 'mean_relative_error', 'std_relative_error']])


,method,kernel,split,mean_relative_error,std_relative_error
0,vanilla,RBF,test,0.047571,0.032136
1,vanilla,RBF,ood,0.074960,0.051710
2,vanilla,Matern,test,0.066662,0.054495
3,vanilla,Matern,ood,0.039798,0.032766
4,product_gpr,Matern x Matern,test,0.017589,0.015206
5,product_gpr,Matern x Matern,ood,0.062367,0.042425
6,product_gpr,RBF x RBF,test,0.029687,0.021464
7,product_gpr,RBF x RBF,ood,0.093622,0.064556


In [5]:
comparison_f2_conservation = compare_to_reference(
    runner_f2_conservation,
    'Framework2/results_Conservation_law_sample_no_pca.csv',
)
display(comparison_f2_conservation)
show_parameter_comparison(comparison_f2_conservation)


,method,kernel,split,mean_relative_error_runner,std_relative_error_runner,length_scale_runner,nu_runner,alpha_runner,coef_length_scale_runner,coef_nu_runner,state_length_scale_runner,state_nu_runner,n_train_runner,n_eval_runner,pca_runner,mean_relative_error_reference,std_relative_error_reference,length_scale_reference,nu_reference,alpha_reference,coef_length_scale_reference,coef_nu_reference,state_length_scale_reference,state_nu_reference,n_train_reference,n_eval_reference,pca_reference,mean_relative_error_diff,mean_relative_error_abs_diff,std_relative_error_diff,std_relative_error_abs_diff
0,vanilla,RBF,test,0.047571,0.032136,100.0,NaN,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,0.047571,0.032136,100.0,NaN,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,2.081668e-17,2.081668e-17,6.245005e-17,6.245005e-17
1,vanilla,RBF,ood,0.074960,0.051710,100.0,NaN,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,0.074960,0.051710,100.0,NaN,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,5.551115e-17,5.551115e-17,8.326673e-17,8.326673e-17
2,vanilla,Matern,test,0.066662,0.054495,1.0,2.5,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,0.066662,0.054495,1.0,2.5,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,0.000000e+00,0.000000e+00,4.857226e-17,4.857226e-17
3,vanilla,Matern,ood,0.039798,0.032766,1.0,2.5,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,0.039798,0.032766,1.0,2.5,1.000000e-10,NaN,NaN,NaN,NaN,10000,4000,False,8.326673e-17,8.326673e-17,2.081668e-17,2.081668e-17
4,product_gpr,Matern x Matern,test,0.017589,0.015206,NaN,NaN,1.000000e-10,1.0,2.5,100.0,1.5,10000,4000,False,0.017589,0.015206,NaN,NaN,1.000000e-10,1.0,2.5,100.0,1.5,10000,4000,False,1.734723e-17,1.734723e-17,5.724587e-17,5.724587e-17
5,product_gpr,Matern x Matern,ood,0.062367,0.042425,NaN,NaN,1.000000e-10,1.0,2.5,100.0,1.5,10000,4000,False,0.062367,0.042425,NaN,NaN,1.000000e-10,1.0,2.5,100.0,1.5,10000,4000,False,7.632783e-17,7.632783e-17,6.938894e-17,6.938894e-17
6,product_gpr,RBF x RBF,test,0.029687,0.021464,NaN,NaN,1.000000e-10,1.0,NaN,100.0,NaN,10000,4000,False,0.029687,0.021464,NaN,NaN,1.000000e-10,1.0,NaN,100.0,NaN,10000,4000,False,3.469447e-18,3.469447e-18,3.816392e-17,3.816392e-17
7,product_gpr,RBF x RBF,ood,0.093622,0.064556,NaN,NaN,1.000000e-10,1.0,NaN,100.0,NaN,10000,4000,False,0.093622,0.064556,NaN,NaN,1.000000e-10,1.0,NaN,100.0,NaN,10000,4000,False,2.775558e-17,2.775558e-17,6.938894e-17,6.938894e-17


,method,kernel,split,length_scale_runner,length_scale_reference,nu_runner,nu_reference,alpha_runner,alpha_reference,coef_length_scale_runner,coef_length_scale_reference,coef_nu_runner,coef_nu_reference,state_length_scale_runner,state_length_scale_reference,state_nu_runner,state_nu_reference,n_train_runner,n_train_reference,n_eval_runner,n_eval_reference,pca_runner,pca_reference
0,vanilla,RBF,test,100.0,100.0,NaN,NaN,1.000000e-10,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000,10000,4000,4000,False,False
1,vanilla,RBF,ood,100.0,100.0,NaN,NaN,1.000000e-10,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000,10000,4000,4000,False,False
2,vanilla,Matern,test,1.0,1.0,2.5,2.5,1.000000e-10,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000,10000,4000,4000,False,False
3,vanilla,Matern,ood,1.0,1.0,2.5,2.5,1.000000e-10,1.000000e-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000,10000,4000,4000,False,False
4,product_gpr,Matern x Matern,test,NaN,NaN,NaN,NaN,1.000000e-10,1.000000e-10,1.0,1.0,2.5,2.5,100.0,100.0,1.5,1.5,10000,10000,4000,4000,False,False
5,product_gpr,Matern x Matern,ood,NaN,NaN,NaN,NaN,1.000000e-10,1.000000e-10,1.0,1.0,2.5,2.5,100.0,100.0,1.5,1.5,10000,10000,4000,4000,False,False
6,product_gpr,RBF x RBF,test,NaN,NaN,NaN,NaN,1.000000e-10,1.000000e-10,1.0,1.0,NaN,NaN,100.0,100.0,NaN,NaN,10000,10000,4000,4000,False,False
7,product_gpr,RBF x RBF,ood,NaN,NaN,NaN,NaN,1.000000e-10,1.000000e-10,1.0,1.0,NaN,NaN,100.0,100.0,NaN,NaN,10000,10000,4000,4000,False,False


## Framework1 Conservation Law: Parameter-Set Mode

This should match `Framework1/results_Conservation_law_parameter_set_no_pca.csv` if that CSV was generated with the same explicit parameters below.


In [7]:
framework1_conservation_config = ExperimentConfig(
    task_mode='parameter_set',
    pde_name='Conservation_law',
    base_dir='Framework1',
    results_dir='.',
    data_subdir='dataset_simple',
    data_filename='solutions.h5',
    ood_filename='ood.h5',
    use_ood=True,
    data_key='data',
    coeff_key='coeffs',
    input_time_index=0,
    channel_index=0,
    ics_per_param=20,
    train_size=None,
    train_fraction=0.8,
    alpha=1e-10,
    use_pca=False,
    use_x_pca=True,
    use_coef_pca=True,
    use_y_pca=True,
    y_pca_by_time=True,
    n_x_pca=5,
    n_coef_pca=4,
    n_y_pca=5,
    vanilla_rbf_length_scale=0.1,
    vanilla_matern_length_scale=0.1,
    vanilla_matern_nu=3/2,
    framework1_rbf_length_scale=0.1,
    framework1_matern_length_scale=1,
    framework1_matern_nu=5/2,
    methods=('vanilla_rbf', 'vanilla_matern', 'method_matern', 'method_rbf'),
    save_csv=True,
    results_filename='parity_Framework1_Conservation_law_parameter_set_no_pca.csv',
)

runner_f1_conservation = run_experiment(framework1_conservation_config)
display(runner_f1_conservation[['method', 'kernel', 'split', 'mean_relative_error', 'std_relative_error']])


,method,kernel,split,mean_relative_error,std_relative_error
0,vanilla,RBF,test,0.037187,0.031158
1,vanilla,RBF,ood,0.084814,0.066353
2,vanilla,Matern,test,0.037187,0.031158
3,vanilla,Matern,ood,0.084814,0.066353
4,framework1,Matern,test,0.000145,0.000221
5,framework1,Matern,ood,0.007724,0.015430
6,framework1,RBF,test,0.000159,0.000234
7,framework1,RBF,ood,0.054740,0.064025


In [8]:
comparison_f1_conservation = compare_to_reference(
    runner_f1_conservation,
    'Framework1/results_Conservation_law_parameter_set_no_pca.csv',
)
display(comparison_f1_conservation)
show_parameter_comparison(comparison_f1_conservation)


,method,kernel,split,mean_relative_error_runner,std_relative_error_runner,length_scale_runner,nu_runner,alpha_runner,n_train_runner,n_eval_runner,pca_runner,mean_relative_error_reference,std_relative_error_reference,length_scale_reference,nu_reference,alpha_reference,n_train_reference,n_eval_reference,pca_reference,mean_relative_error_diff,mean_relative_error_abs_diff,std_relative_error_diff,std_relative_error_abs_diff
0,vanilla,RBF,test,0.037187,0.031158,0.1,NaN,1.000000e-10,6400,1600,False,0.037187,0.031158,0.1,NaN,1.000000e-10,6400,1600,False,4.163336e-17,4.163336e-17,3.469447e-18,3.469447e-18
1,vanilla,RBF,ood,0.084814,0.066353,0.1,NaN,1.000000e-10,6400,800,False,0.084814,0.066353,0.1,NaN,1.000000e-10,6400,800,False,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,vanilla,Matern,test,0.037187,0.031158,0.1,1.5,1.000000e-10,6400,1600,False,0.037187,0.031158,0.1,1.5,1.000000e-10,6400,1600,False,2.775558e-17,2.775558e-17,6.938894e-17,6.938894e-17
3,vanilla,Matern,ood,0.084814,0.066353,0.1,1.5,1.000000e-10,6400,800,False,0.084814,0.066353,0.1,1.5,1.000000e-10,6400,800,False,5.551115e-17,5.551115e-17,6.938894e-17,6.938894e-17
4,framework1,Matern,test,0.000145,0.000221,1.0,2.5,1.000000e-10,320,1600,False,0.000145,0.000221,1.0,2.5,1.000000e-10,320,1600,False,6.104058e-17,6.104058e-17,7.817098e-17,7.817098e-17
5,framework1,Matern,ood,0.007724,0.015430,1.0,2.5,1.000000e-10,320,800,False,0.007724,0.015430,1.0,2.5,1.000000e-10,320,800,False,2.949030e-17,2.949030e-17,1.734723e-17,1.734723e-17
6,framework1,RBF,test,0.000159,0.000234,0.1,NaN,1.000000e-10,320,1600,False,0.000159,0.000234,0.1,NaN,1.000000e-10,320,1600,False,7.624652e-17,7.624652e-17,9.576216e-17,9.576216e-17
7,framework1,RBF,ood,0.054740,0.064025,0.1,NaN,1.000000e-10,320,800,False,0.054740,0.064025,0.1,NaN,1.000000e-10,320,800,False,5.551115e-17,5.551115e-17,6.938894e-17,6.938894e-17


,method,kernel,split,length_scale_runner,length_scale_reference,nu_runner,nu_reference,alpha_runner,alpha_reference,n_train_runner,n_train_reference,n_eval_runner,n_eval_reference,pca_runner,pca_reference
0,vanilla,RBF,test,0.1,0.1,NaN,NaN,1.000000e-10,1.000000e-10,6400,6400,1600,1600,False,False
1,vanilla,RBF,ood,0.1,0.1,NaN,NaN,1.000000e-10,1.000000e-10,6400,6400,800,800,False,False
2,vanilla,Matern,test,0.1,0.1,1.5,1.5,1.000000e-10,1.000000e-10,6400,6400,1600,1600,False,False
3,vanilla,Matern,ood,0.1,0.1,1.5,1.5,1.000000e-10,1.000000e-10,6400,6400,800,800,False,False
4,framework1,Matern,test,1.0,1.0,2.5,2.5,1.000000e-10,1.000000e-10,320,320,1600,1600,False,False
5,framework1,Matern,ood,1.0,1.0,2.5,2.5,1.000000e-10,1.000000e-10,320,320,800,800,False,False
6,framework1,RBF,test,0.1,0.1,NaN,NaN,1.000000e-10,1.000000e-10,320,320,1600,1600,False,False
7,framework1,RBF,ood,0.1,0.1,NaN,NaN,1.000000e-10,1.000000e-10,320,320,800,800,False,False


## Notes

If the error values differ but the parameter-comparison table matches, the difference is likely from implementation details rather than hyperparameters. If the parameter-comparison table differs, update the explicit config above to match the reference CSV/notebook exactly.


In [9]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 586.0 kB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [optuna]2m4/5 [optuna]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
